# Historical reproduction: coarse seed-opportunity bracket v1

This notebook is a readable reconstruction of one completed, audited local experiment. It does **not** rerun Kaggriculture, modify policy source, invoke a model, or provision a cloud resource.

The adjacent JSON file is canonical. This notebook loads it rather than copying its values into code, so the human walkthrough and machine contract stay connected.

## 1. What was being tested?

The carrot-investment graph compares an expected carrot return with the value of preserving an owned seed for a later opportunity. `seed_opportunity_value` was the static shadow price of that preserved option.

The question was narrow: with all game facts and controls held fixed, do selected static values improve or harm whole-season reward against the deterministic carrot conveyor? This is not a claim about global farm strategy or a context-aware value supplied by future decision graphs.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path


SPEC_NAME = "seed_opportunity_coarse_v1.json"


def locate_spec() -> Path:
    """Find the adjacent spec whether Jupyter starts here or at project root."""
    relative = Path("carrot-tuning-system/experiments/historical/carrot_investment") / SPEC_NAME
    for root in (Path.cwd(), *Path.cwd().parents):
        direct = root / SPEC_NAME
        nested = root / relative
        if direct.is_file():
            return direct
        if nested.is_file():
            return nested
    raise FileNotFoundError(f"Could not find {SPEC_NAME} from {Path.cwd()}")


spec_path = locate_spec()
spec = json.loads(spec_path.read_text())
print(f"Loaded canonical historical spec: {spec_path}")
print(f"Status: {spec['lifecycle_status']}")

## 2. Controls that make the comparison meaningful

Every candidate used the same target fingerprint, deterministic opponent, seeds, seats, turn limit, and baseline. The only intended variation was the one in-memory policy field declared in the specification.

In [ ]:
suite = spec['suite']
parameter = spec['parameter']
target = spec['target']

print(f"Decision graph: {spec['decision_graph']['id']}")
print(f"Parameter: {parameter['name']} (baseline {parameter['baseline_value']})")
print(f"Candidates: {parameter['candidate_values']}")
print(f"Opponent: {suite['opponent_id']} (deterministic={suite['opponent_deterministic']})")
print(f"Matched suite: {len(suite['seeds'])} seeds × {len(suite['seats'])} seats × {suite['steps']} turns")
print(f"Target revision: {target['expected_revision']}")

## 3. Why a fast result is still trustworthy

The baseline saved simulator state immediately before every decision. A candidate replayed baseline observations until its first different action. No divergence meant exact baseline reuse; a divergence meant restore the saved pre-action state and simulate the remaining suffix.

Every one of those fast results was also checked by a fresh, clean full candidate rerun. The audit is the authority: an audit mismatch would invalidate the cached result.

## 4. Recorded aggregate evidence

These values come from `historical_expectations` in the canonical spec. They are regression expectations for a future inner-loop runner, not a new execution from this notebook.

In [ ]:
expected = spec['historical_expectations']
print(f"Baseline mean reward: {expected['baseline_mean_reward']:.2f}")
print(f"All clean audits passed: {expected['all_clean_audits_passed']}")
print()
for row in expected['candidates']:
    print(
        f"value={row['value']:>5.0f} | reward={row['mean_reward']:>7.2f} "
        f"| delta={row['mean_paired_reward_delta']:>8.2f} "
        f"| {row['classification']} | audits={row['clean_audits_passed']}"
    )

## 5. Interpretation and limits

Value 10 was inert: it did not cross a realized decision threshold. Values 40, 60, and 80 progressively reduced reward, consistent with suppressing profitable planting. The supported conclusion is narrow: retain static value 20 for this exact conveyor suite.

This does not prove 20 is universally optimal. A future crop-selection, land-allocation, or higher-level planning graph may provide a context-dependent opportunity value. That future input must be tested with the same matched, clean-audited process.

## 6. Optional raw-result lookup

The generated raw report is intentionally ignored and may not be present in a fresh clone. If it is available locally, this cell reports its location. The notebook remains useful without it because the source-controlled spec retains the contract and expected aggregate findings.

In [ ]:
project_root = spec_path.parents[4]
raw_result = project_root / 'var/results/coarse-preferences-60/summary.json'
if raw_result.is_file():
    raw = json.loads(raw_result.read_text())
    print(f"Optional raw result is available: {raw_result}")
    print(f"Recorded wall time: {raw['wall_elapsed_seconds']:.1f} seconds")
else:
    print("Optional raw result is not present. That is expected for a disposable ignored artifact.")

## 7. How this becomes a smoke test

The first real inner-loop runner should accept this record only when explicitly approved for reproduction, create or validate the matching baseline cache, execute precisely the declared candidates, and compare its resulting classifications with this historical expectation. If target hashes, clean audits, or classifications disagree, stop and repair the runner before adding Foundry reasoning or a new decision graph.